# Python Package Manager Exercises:


1. Read this url and find the 10 most frequent words. romeo_and_juliet = 'http://www.gutenberg.org/files/1112/1112.txt'


In [4]:
import re
from collections import Counter
from urllib.request import Request, urlopen

def read_url(url):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8", errors="replace")

# Gutenberg plain-text download
url = "https://www.gutenberg.org/cache/epub/1112/pg1112.txt"

fallback_urls = [
    url,
    "https://www.gutenberg.org/files/1112/1112.txt",
    "https://www.gutenberg.org/files/1112/1112-8.txt",
]

for candidate_url in fallback_urls:
    try:
        text = read_url(candidate_url)
        break
    except Exception as error:
        last_error = error
else:
    fallback_urls = [
        "https://r.jina.ai/http://www.gutenberg.org/files/1112/1112.txt",
        "https://raw.githubusercontent.com/GITenberg/The-Tragedie-of-Romeo-and-Juliet_1112/master/1112.txt",
    ]

    for candidate_url in fallback_urls:
        try:
            text = read_url(candidate_url)
            break
        except Exception as error:
            last_error = error
    else:
        raise RuntimeError(
            "Unable to download Romeo and Juliet; please check your network connection."
        ) from last_error

# Extract words; convert to lowercase so "Romeo" and "romeo" count together.
words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

# Optional: remove Gutenberg's header and footer if present.
start_marker = "*** START OF THE PROJECT GUTENBERG"
end_marker = "*** END OF THE PROJECT GUTENBERG"

start = text.find(start_marker)
end = text.find(end_marker)

if start != -1 and end != -1:
    play_text = text[start:end]
    words = re.findall(r"\b[a-zA-Z]+\b", play_text.lower())

most_common_words = Counter(words).most_common(10)

print("10 most frequent words:")
for count, word in most_common_words:
    print(word, count)

10 most frequent words:
5 gutenberg
4 https
3 url
3 www
3 org
2 temporarily
2 for
2 maintenance
2 files
2 txt


2. Read the cats API and cats_api = 'https://api.thecatapi.com/v1/breeds' and find :


In [ ]:
from collections import Counter
from statistics import mean, median, pstdev
from urllib.request import Request, urlopen
import json
import re
import os
from getpass import getpass

def get_json(url):
    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))

cats_api = "https://api.thecatapi.com/v1/breeds"
cats = None
last_error = None

fallback_urls = [
    cats_api,
    "https://r.jina.ai/http://api.thecatapi.com/v1/breeds",
    "https://api.allorigins.win/raw?url=https%3A%2F%2Fapi.thecatapi.com%2Fv1%2Fbreeds",
]

fallback_urls = [
    "https://api.thecatapi.com/v1/breeds?limit=100",
    "https://api.thecatapi.com/v1/breeds",
    "https://corsproxy.io/?url=https%3A%2F%2Fapi.thecatapi.com%2Fv1%2Fbreeds",
]

api_key = os.environ.get("THECATAPI_KEY") or getpass("Enter your TheCatAPI key: ")

# Use the API directly; proxy URLs may cause authorization failures.
fallback_urls = [
    "https://api.thecatapi.com/v1/breeds?limit=100",
    cats_api,
]

for candidate_url in fallback_urls:
    try:
        request = Request(
            candidate_url,
            headers={
                "User-Agent": "Mozilla/5.0",
                "x-api-key": api_key,
            },
        )
        with urlopen(request, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))

        if isinstance(data, list) and data:
            cats = data
            break
    except Exception as error:
        last_error = error

if cats is None:
    raise RuntimeError("Unable to retrieve cat breed data.") from last_error

# Extract numeric values from ranges such as "14-17".
def range_midpoint(value):
    numbers = re.findall(r"\d+(?:\.\d+)?", str(value))

    if not numbers:
        return None

    numbers = [float(number) for number in numbers]
    return mean(numbers)

# Collect weight and lifespan values.
weights = []
lifespans = []

for cat in cats:
    weight = cat.get("weight", {}).get("metric")
    lifespan = cat.get("life_span")

    weight_value = range_midpoint(weight)
    lifespan_value = range_midpoint(lifespan)

    if weight_value is not None:
        weights.append(weight_value)

    if lifespan_value is not None:
        lifespans.append(lifespan_value)


def describe(values):
    return {
        "min": min(values),
        "max": max(values),
        "mean": mean(values),
        "median": median(values),
        "standard_deviation": pstdev(values)
    }


print("CAT WEIGHT (kg)")
print(describe(weights))

print("\nCAT LIFESPAN (years)")
print(describe(lifespans))


# Frequency table: country of origin and breed.
country_breed_counts = Counter(
    (cat.get("origin", "Unknown"), cat.get("name", "Unknown"))
    for cat in cats
)

print("\nCOUNTRY AND BREED FREQUENCY TABLE")
print("Country | Breed | Frequency")

for (country, breed), count in sorted(country_breed_counts.items()):
    print(country, "|", breed, "|", count)

    i. the min, max, mean, median, standard deviation of cats' weight in metric units.


    ii. the min, max, mean, median, standard deviation of cats' lifespan in years.


    iii. Create a frequency table of country and breed of cats


3. Read the countries API and find


    i. the 10 largest countries


    ii. the 10 most spoken languages


    iii. the total number of languages in the countries API


4. UCI is one of the most common places to get data sets for data science and machine learning. Read the content of UCL (https://archive.ics.uci.edu/ml/datasets.php). Without additional libraries it will be difficult, so you may try it with BeautifulSoup4